# Evaluación de modelos — métricas

Evaluar un modelo no es probarlo a ojo — es medirlo con números concretos
sobre ejemplos que nunca ha visto durante el entrenamiento.

## Conceptos clave

**Test set** — conjunto de ejemplos reservados exclusivamente para evaluación.
El modelo nunca los ve durante el entrenamiento. Si usas los mismos datos
para entrenar y evaluar, los resultados no son fiables.

**Accuracy** — porcentaje de predicciones correctas sobre el total.
Simple pero engañosa: si el 90% de tickets son P1, un modelo que
siempre dice P1 tiene 90% de accuracy sin aprender nada.

**Precision** — de todos los que predijo como P1, ¿cuántos eran realmente P1?
Mide falsos positivos.

**Recall** — de todos los P1 reales, ¿cuántos detectó el modelo?
Mide falsos negativos.

**F1** — media armónica entre precision y recall.
Es la métrica más equilibrada para clasificadores con clases desbalanceadas.

**Confusion matrix** — tabla que muestra exactamente dónde se equivoca el modelo.
Filas = etiquetas reales, columnas = predicciones del modelo.

## Dataset de evaluación

Creamos un test set separado del dataset de entrenamiento.
Estos ejemplos el modelo nunca los ha visto — son la prueba real.

En un proyecto real el split sería automático:
80% entrenamiento, 20% evaluación.
Con 8 ejemplos era demasiado pequeño para dividir,
por eso creamos el test set manualmente.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
import numpy as np

# Test set — ejemplos nuevos con etiquetas reales conocidas
test_data = [
    {"ticket": "Database server completely unresponsive", "expected": "P1"},
    {"ticket": "Wrong icon displayed in settings menu", "expected": "P4"},
    {"ticket": "API response time degraded to 3 seconds", "expected": "P2"},
    {"ticket": "Login page crashes on Safari mobile", "expected": "P3"},
    {"ticket": "All microservices down, complete outage", "expected": "P1"},
    {"ticket": "Tooltip text has a typo", "expected": "P4"},
    {"ticket": "Payment processing delayed by 30 seconds", "expected": "P2"},
    {"ticket": "Memory leak causing gradual service degradation", "expected": "P2"},
]

print(f"Test set: {len(test_data)} ejemplos")
for d in test_data:
    print(f"  [{d['expected']}] {d['ticket']}")

## Setup — importar utilidades compartidas

Las funciones de clasificación viven en `utils/classifier.py`
para poder reutilizarlas desde cualquier notebook sin duplicar código.

In [18]:
import sys
import os
print(os.getcwd())
sys.path.append("..")
from utils.classifier import cargar_modelo, clasificar_ticket, extraer_severidad

model, tokenizer = cargar_modelo(
    lora_path="../03_finetuning/lora_output/checkpoint-80"
)

/Users/alex/Projects/AI_Playground/notebooks/04_eval


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 500.31it/s]


## Predicciones del modelo

Ejecutamos el modelo fine-tuneado sobre cada ejemplo del test set
y extraemos la severidad predicha para compararla con la real.

In [20]:
etiquetas_reales = []
etiquetas_predichas = []

for ejemplo in test_data:
    prediccion_raw = clasificar_ticket(ejemplo["ticket"], model, tokenizer)
    prediccion = extraer_severidad(prediccion_raw)

    etiquetas_reales.append(ejemplo["expected"])
    etiquetas_predichas.append(prediccion)

    correcto = "✅" if prediccion == ejemplo["expected"] else "❌"
    print(f"{correcto} Real: {ejemplo['expected']} | Predicho: {prediccion} | {ejemplo['ticket']}")

✅ Real: P1 | Predicho: P1 | Database server completely unresponsive
❌ Real: P4 | Predicho: P1 | Wrong icon displayed in settings menu
❌ Real: P2 | Predicho: P1 | API response time degraded to 3 seconds
❌ Real: P3 | Predicho: P1 | Login page crashes on Safari mobile
❌ Real: P1 | Predicho: UNKNOWN | All microservices down, complete outage
❌ Real: P4 | Predicho: P2 | Tooltip text has a typo
✅ Real: P2 | Predicho: P2 | Payment processing delayed by 30 seconds
✅ Real: P2 | Predicho: P2 | Memory leak causing gradual service degradation


## Métricas formales

Con las predicciones recopiladas calculamos las métricas reales.
Esto convierte la intuición ("falla mucho en P1") en números concretos
que puedes comparar entre versiones del modelo.

In [21]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

# Classification report — precision, recall y F1 por clase
print("CLASSIFICATION REPORT:")
print(classification_report(
    etiquetas_reales,
    etiquetas_predichas,
    labels=["P1", "P2", "P3", "P4"],
    zero_division=0
))

# Confusion matrix
cm = confusion_matrix(
    etiquetas_reales,
    etiquetas_predichas,
    labels=["P1", "P2", "P3", "P4"]
)

df_cm = pd.DataFrame(
    cm,
    index=["Real P1", "Real P2", "Real P3", "Real P4"],
    columns=["Pred P1", "Pred P2", "Pred P3", "Pred P4"]
)

print("\nCONFUSION MATRIX:")
print(df_cm)

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

          P1       0.25      0.50      0.33         2
          P2       0.67      0.67      0.67         3
          P3       0.00      0.00      0.00         1
          P4       0.00      0.00      0.00         2

   micro avg       0.43      0.38      0.40         8
   macro avg       0.23      0.29      0.25         8
weighted avg       0.31      0.38      0.33         8


CONFUSION MATRIX:
         Pred P1  Pred P2  Pred P3  Pred P4
Real P1        1        0        0        0
Real P2        1        2        0        0
Real P3        1        0        0        0
Real P4        1        1        0        0


## Conclusiones de la evaluación

**Accuracy: 37.5%** — el modelo acierta menos de la mitad.

**El problema es claro en la confusion matrix:**
el modelo predice P1 en casi todos los casos.
Es el overfitting que identificamos antes — con solo 8 ejemplos
de entrenamiento el modelo aprendió a decir P1 por defecto.

**Qué necesitaría para mejorar:**
- Dataset de 200-500 ejemplos balanceados (50 por clase mínimo)
- Early stopping para evitar memorización
- Validación durante el entrenamiento para detectar overfitting a tiempo

**El valor de este ejercicio:**
Hemos completado el ciclo completo de fine-tuning:
dataset → entrenamiento → evaluación → diagnóstico.
Las métricas nos dicen exactamente qué hay que mejorar
y cuánto — algo imposible de saber "a ojo".